In [6]:
import pandas as pd
import numpy as np
import json

data = {
    'product_id': [1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 5, 5, 6, 6, 7, 8, 8],
    'product_name': ['Wireless Mouse', 'Wireless Mouse', 'Wireless Mouse', 'Mechanical Keyboard', 'Mechanical Keyboard', 'Mechanical Keyboard', 'USB-C Hub', 'USB-C Hub', 'USB-C Hub', 'Laptop Stand', 'Laptop Stand', 'HDMI Cable 2m', 'HDMI Cable 2m', 'Webcam 1080p', 'Webcam 1080p', 'Desk Lamp LED', 'Ergonomic Chair', 'Ergonomic Chair'],
    'warehouse_id': [1, 2, 3, 1, 2, 3, 1, 2, 3, 1, 2, 1, 2, 1, 2, 1, 1, 2],
    'warehouse_name': ['Warehouse Alpha', 'Warehouse Beta', 'Warehouse Gamma', 'Warehouse Alpha', 'Warehouse Beta', 'Warehouse Gamma', 'Warehouse Alpha', 'Warehouse Beta', 'Warehouse Gamma', 'Warehouse Alpha', 'Warehouse Beta', 'Warehouse Alpha', 'Warehouse Beta', 'Warehouse Alpha', 'Warehouse Beta', 'Warehouse Alpha', 'Warehouse Alpha', 'Warehouse Beta'],
    'movement_type': ['IN', 'IN', 'IN', 'IN', 'OUT', 'OUT', 'IN', 'IN', 'OUT', 'IN', 'IN', 'OUT', 'OUT', 'IN', 'ADJUSTMENT', 'IN', 'IN', 'OUT'],
    'quantity': [120, 80, 60, 50, 42, 30, 45, 30, 25, 400, 380, 25, 8, 24, 20, 55, 3, 1],
    'movement_date': ['2024-01-10', '2024-01-10', '2024-01-10', '2024-01-12', '2024-01-13', '2024-01-14', '2024-01-11', '2024-01-12', '2024-01-15', '2024-01-08', '2024-01-09', '2024-01-14', '2024-01-15', '2024-01-16', '2024-01-16', '2024-01-13', '2024-01-17', '2024-01-18']
}

df = pd.read_json(json.dumps(data))
print("Raw Data Loaded:")
print(df)

Raw Data Loaded:
    product_id         product_name  warehouse_id   warehouse_name  \
0            1       Wireless Mouse             1  Warehouse Alpha   
1            1       Wireless Mouse             2   Warehouse Beta   
2            1       Wireless Mouse             3  Warehouse Gamma   
3            2  Mechanical Keyboard             1  Warehouse Alpha   
4            2  Mechanical Keyboard             2   Warehouse Beta   
5            2  Mechanical Keyboard             3  Warehouse Gamma   
6            3            USB-C Hub             1  Warehouse Alpha   
7            3            USB-C Hub             2   Warehouse Beta   
8            3            USB-C Hub             3  Warehouse Gamma   
9            4         Laptop Stand             1  Warehouse Alpha   
10           4         Laptop Stand             2   Warehouse Beta   
11           5        HDMI Cable 2m             1  Warehouse Alpha   
12           5        HDMI Cable 2m             2   Warehouse Beta   
13 

/tmp/ipykernel_996/2547063273.py:15: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_json(json.dumps(data))


In [7]:
df['movement_date'] = pd.to_datetime(df['movement_date'])
df['quantity'] = df['quantity'].astype(int)
df = df[df['quantity'] != 0]
valid_types = ['IN', 'OUT', 'ADJUSTMENT']
df = df[df['movement_type'].isin(valid_types)]
print("Cleaned Data:")
print(df)
print(f"\nTotal records after cleaning: {len(df)}")

Cleaned Data:
    product_id         product_name  warehouse_id   warehouse_name  \
0            1       Wireless Mouse             1  Warehouse Alpha   
1            1       Wireless Mouse             2   Warehouse Beta   
2            1       Wireless Mouse             3  Warehouse Gamma   
3            2  Mechanical Keyboard             1  Warehouse Alpha   
4            2  Mechanical Keyboard             2   Warehouse Beta   
5            2  Mechanical Keyboard             3  Warehouse Gamma   
6            3            USB-C Hub             1  Warehouse Alpha   
7            3            USB-C Hub             2   Warehouse Beta   
8            3            USB-C Hub             3  Warehouse Gamma   
9            4         Laptop Stand             1  Warehouse Alpha   
10           4         Laptop Stand             2   Warehouse Beta   
11           5        HDMI Cable 2m             1  Warehouse Alpha   
12           5        HDMI Cable 2m             2   Warehouse Beta   
13    

In [8]:
stock_data = []
for product_id in df['product_id'].unique():
    for warehouse_id in df[df['product_id'] == product_id]['warehouse_id'].unique():
        product_data = df[(df['product_id'] == product_id) & (df['warehouse_id'] == warehouse_id)]
        in_total = product_data[product_data['movement_type'] == 'IN']['quantity'].sum()
        out_total = product_data[product_data['movement_type'] == 'OUT']['quantity'].sum()
        adj_total = product_data[product_data['movement_type'] == 'ADJUSTMENT']['quantity'].sum()
        current_stock = in_total - out_total + adj_total
        product_name = product_data.iloc[0]['product_name']
        warehouse_name = product_data.iloc[0]['warehouse_name']
        stock_data.append({
            'product_id': product_id,
            'product_name': product_name,
            'warehouse_id': warehouse_id,
            'warehouse_name': warehouse_name,
            'current_stock': current_stock
        })
stock_df = pd.DataFrame(stock_data)
print("Current Stock Levels:")
print(stock_df)

Current Stock Levels:
    product_id         product_name  warehouse_id   warehouse_name  \
0            1       Wireless Mouse             1  Warehouse Alpha   
1            1       Wireless Mouse             2   Warehouse Beta   
2            1       Wireless Mouse             3  Warehouse Gamma   
3            2  Mechanical Keyboard             1  Warehouse Alpha   
4            2  Mechanical Keyboard             2   Warehouse Beta   
5            2  Mechanical Keyboard             3  Warehouse Gamma   
6            3            USB-C Hub             1  Warehouse Alpha   
7            3            USB-C Hub             2   Warehouse Beta   
8            3            USB-C Hub             3  Warehouse Gamma   
9            4         Laptop Stand             1  Warehouse Alpha   
10           4         Laptop Stand             2   Warehouse Beta   
11           5        HDMI Cable 2m             1  Warehouse Alpha   
12           5        HDMI Cable 2m             2   Warehouse Beta  

In [9]:
reorder_level = 10
low_stock = stock_df[stock_df['current_stock'] <= reorder_level]

print("Low Stock Items (Below Reorder Level):")
print(low_stock[['product_name', 'warehouse_name', 'current_stock']])
print(f"\nTotal items below reorder level: {len(low_stock)}")

Low Stock Items (Below Reorder Level):
           product_name   warehouse_name  current_stock
4   Mechanical Keyboard   Warehouse Beta            -42
5   Mechanical Keyboard  Warehouse Gamma            -30
8             USB-C Hub  Warehouse Gamma            -25
11        HDMI Cable 2m  Warehouse Alpha            -25
12        HDMI Cable 2m   Warehouse Beta             -8
16      Ergonomic Chair  Warehouse Alpha              3
17      Ergonomic Chair   Warehouse Beta             -1

Total items below reorder level: 7


In [10]:
summary = {
    'total_products': int(stock_df['product_id'].nunique()),
    'total_warehouses': int(stock_df['warehouse_id'].nunique()),
    'total_stock_across_all': int(stock_df['current_stock'].sum()),
    'reorder_level': reorder_level,
    'items_below_reorder': int(len(low_stock)),
    'low_stock_items': low_stock.to_dict('records')
}

print("=== SUMMARY REPORT ===")
print(json.dumps(summary, indent=2))

=== SUMMARY REPORT ===
{
  "total_products": 8,
  "total_warehouses": 3,
  "total_stock_across_all": 1136,
  "reorder_level": 10,
  "items_below_reorder": 7,
  "low_stock_items": [
    {
      "product_id": 2,
      "product_name": "Mechanical Keyboard",
      "warehouse_id": 2,
      "warehouse_name": "Warehouse Beta",
      "current_stock": -42
    },
    {
      "product_id": 2,
      "product_name": "Mechanical Keyboard",
      "warehouse_id": 3,
      "warehouse_name": "Warehouse Gamma",
      "current_stock": -30
    },
    {
      "product_id": 3,
      "product_name": "USB-C Hub",
      "warehouse_id": 3,
      "warehouse_name": "Warehouse Gamma",
      "current_stock": -25
    },
    {
      "product_id": 5,
      "product_name": "HDMI Cable 2m",
      "warehouse_id": 1,
      "warehouse_name": "Warehouse Alpha",
      "current_stock": -25
    },
    {
      "product_id": 5,
      "product_name": "HDMI Cable 2m",
      "warehouse_id": 2,
      "warehouse_name": "Warehouse Beta

In [11]:
with open('stock_summary_report.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Report saved to stock_summary_report.json")
print("\nDownload the file from the sidebar")

Report saved to stock_summary_report.json

Download the file from the sidebar
